# PFE ML — Run 9: Gradient Boosted Trees

Same data and the same 2M-row cap as Run 8. The only thing that changes is the classifier.

## What's new on the branch

`app/tools/train_continuity_model.py` now uses `HistGradientBoostingClassifier` instead of `LogisticRegression`:

- **No imputer, no scaler, no one-hot encoder.** HGB handles NaN natively (separate missing-value bin) and accepts pandas `Categorical` columns directly via `categorical_features="from_dtype"`. The 1,703 distinct `activity_code` values are capped to the top-250 by a small `CategoricalCardinalityCapper` step so the column fits inside HGB's `max_bins` limit; the rest fold into `__OTHER__`.
- **`latest_equity_ratio` is excluded** (perfect collinearity with `latest_debt_to_assets`, corr 0.9998 in the run 8 audit). Same accounting identity, no extra signal, hurts importance attribution.
- **`feature_coefficients.csv` is replaced by `feature_importances.csv`** computed via permutation importance on a 50k-row test slice. ROC-AUC drop is the scoring metric. The plot is now `top_feature_importances.png`.
- **Run folder slug uses `hgb`** instead of `logreg`, so chronologically sorted runs make the model-family switch visible at a glance.

Realistic expectation: ROC-AUC moves from 0.7475 to 0.80+, average precision roughly doubles, top-K lift jumps. Trees are the natural fit for high-cardinality categoricals and heavy missingness.

## 1. Runtime

**High-RAM CPU.** Training ~5–8 min, permutation importance ~3 min, plots and comparison ~1 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('BRANCH       =', BRANCH)
print('TRAIN_CAP    =', TRAIN_MAX_ROWS)

## 2. Pull Code And Install Dependencies

Commit and push the HGB switch on `data-extraction` before running — otherwise the pull below brings in the Run 8 logistic-regression code and you retrain Run 8.

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
train_script_text = (Path(BACKEND_DIR) / 'app' / 'tools' / 'train_continuity_model.py').read_text(encoding='utf-8')
checks = {
    'HistGradientBoostingClassifier imported': 'HistGradientBoostingClassifier' in train_script_text,
    'CategoricalCardinalityCapper present': 'CategoricalCardinalityCapper' in train_script_text,
    'latest_equity_ratio excluded': 'latest_equity_ratio' in train_script_text.split('def main')[0],
    'permutation_importance used': 'permutation_importance' in train_script_text,
    'logreg model_family is gone': 'model_family="logreg"' not in train_script_text,
}
print('Branch readiness:')
for label, ok in checks.items():
    print(f'  {"OK " if ok else "FAIL"}  {label}')
if not all(checks.values()):
    raise SystemExit('HGB switch is missing on the pulled branch. Commit + push the train_continuity_model.py changes and re-run this cell.')

## 3. Verify Clean Layer And Feature Table Are Still On Drive

Same period-fixed features Run 8 used. We're not rebuilding anything.

In [ ]:
import json, duckdb

manifest_path = Path(f'{DATA_LAKE}/clean/company_identity/_manifest.json')
if not manifest_path.exists():
    raise SystemExit(f'Missing manifest at {manifest_path}. Run pfe_ml_colab_period_fix_rebuild.ipynb first.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest.get('schema_version') == 2 and 'period' in manifest.get('grain', ''), (
    f'Clean layer is not period-fixed: schema_version={manifest.get("schema_version")!r}, grain={manifest.get("grain")!r}'
)

features_root = Path(f'{DATA_LAKE}/features/company_year_features')
labels_root = Path(f'{DATA_LAKE}/features/risk_labels')
feature_files = list(features_root.rglob('*.parquet')) if features_root.exists() else []
label_files = list(labels_root.rglob('*.parquet')) if labels_root.exists() else []
if not feature_files or not label_files:
    raise SystemExit('Feature or label parquet missing — run the period-fix rebuild notebook first.')

con = duckdb.connect()
counts = con.execute(f'''
    SELECT
        (SELECT count(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS feature_rows,
        (SELECT count(*) FROM read_parquet('{LABELS_GLOB}',   union_by_name=true)) AS label_rows,
        (SELECT min(prediction_year) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS min_year,
        (SELECT max(prediction_year) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS max_year
''').df()
print('Feature/label counts:')
print(counts.to_string(index=False))
con.close()

## 4. Train Run 9 (HistGradientBoostingClassifier)

Same 2M-row cap as Runs 1, 2, 7, 8. Only the classifier changed. Any deltas in metrics are attributable to the model family. The script prints permutation-importance progress; expect 8–12 min total.

In [ ]:
import shlex, subprocess, sys

train_cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.train_continuity_model',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
    '--target', TARGET,
    '--train-start-year', str(START_YEAR),
    '--train-end-year', str(END_YEAR),
    '--max-rows', str(TRAIN_MAX_ROWS),
    '--min-rows', '1000',
]
print(' '.join(shlex.quote(p) for p in train_cmd))
subprocess.run(train_cmd, check=True)

metadata_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
print('\n' + json.dumps({
    'run_name': metadata.get('run_name'),
    'model_version': metadata.get('model_version'),
    'rows': metadata.get('rows'),
    'feature_count': metadata.get('feature_count'),
    'excluded_columns_present': metadata.get('excluded_columns_present'),
    'split_strategy': metadata.get('split_strategy'),
    'test_class_counts': metadata.get('test_class_counts'),
    'metrics': {
        k: metadata.get('metrics', {}).get(k)
        for k in ('accuracy', 'roc_auc', 'average_precision',
                  'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5')
    },
    'run_artifacts_dir': metadata.get('run_artifacts_dir'),
}, indent=2))

## 5. Display Run Artifacts

In [ ]:
from IPython.display import Image, Markdown, display

run_dir = Path(metadata['run_artifacts_dir'])
print('Run folder:', run_dir)
print('\nFiles:')
for path in sorted(run_dir.iterdir()):
    print(' ', path.name)

summary_path = run_dir / 'run_summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))

image_names = [
    'class_counts_by_year.png',
    'precision_recall_curve.png',
    'roc_curve.png',
    'confusion_matrix_at_0_5.png',
    'score_distribution_by_class.png',
    'threshold_tradeoff.png',
    'top_feature_importances.png',
]
for image_name in image_names:
    image_path = run_dir / image_name
    if image_path.exists():
        print('\n' + image_name)
        display(Image(filename=str(image_path)))

comparison_image = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.png'
if comparison_image.exists():
    print('\nmodel_run_comparison.png')
    display(Image(filename=str(comparison_image)))

## 6. Compare All Runs

Side-by-side of every run (runs 1–8 logistic regression, run 9 HGB). The first three columns identify the run; the rest are the comparable test metrics.

In [ ]:
import pandas as pd

comparison_csv = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.csv'
if comparison_csv.exists():
    runs_summary = pd.read_csv(comparison_csv)
    keep = [
        'run_name', 'rows', 'feature_count', 'test_positive_rate',
        'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5',
    ]
    print('All runs (chronological):')
    print(runs_summary[keep].to_string(index=False))
    print('\nLast row is Run 9 (HGB). Compare especially: roc_auc, average_precision, f1_at_0_5.')
else:
    print('No model_run_comparison.csv yet.')

In [ ]:
runs_dir = Path(DRIVE_ROOT) / 'ml-artifacts' / 'runs'
run_folders = sorted(runs_dir.iterdir(), key=lambda p: p.name)
if len(run_folders) >= 2:
    new_dir = run_folders[-1]
    prev_dir = run_folders[-2]
    print(f'Previous run (Run 8, logreg): {prev_dir.name}')
    print(f'New run (Run 9, hgb):         {new_dir.name}\n')

    new_importances_csv = new_dir / 'feature_importances.csv'
    prev_coefficients_csv = prev_dir / 'feature_coefficients.csv'

    if new_importances_csv.exists():
        new_imp = pd.read_csv(new_importances_csv).head(20)
        print('Run 9 — top 20 features by permutation importance (ROC-AUC drop):')
        print(new_imp.to_string(index=False))
    if prev_coefficients_csv.exists():
        prev_coef = pd.read_csv(prev_coefficients_csv).head(20)
        print('\nRun 8 — top 20 features by abs(coefficient) (for reference):')
        print(prev_coef[['feature', 'coefficient']].to_string(index=False))

## 7. What To Send After The Run

- The newest folder under `ml-artifacts/runs/` (the `_hgb_*` one): `run_summary.md`, `feature_importances.csv`, `top_feature_importances.png`, `precision_recall_curve.png`, `roc_curve.png`, `threshold_tradeoff.png`.
- `ml-artifacts/model_run_comparison.png` and `.csv` (now 9 rows).
- Note in the report: AUC, average precision, and top-K lift before vs after the model-family switch — those four numbers are the headline.
- The permutation-importance ranking (section 6) shows which features the tree model actually relies on, after the linear model's `activity_code_*` one-hots are no longer dominating the top of the list.